<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding #4 — The Freshness Multiplier (p. 9).** The paper reports a 361+ day freshness bucket with a 283:1 growth-to-decline ratio, and to its credit the paper itself flags this as unstable, since it comes from 283 growing pages against a single declining page in that bucket.

My methodology question: every ratio in this table separates pages into "growing" and "declining" using `trend_direction`, which the paper's own "How to Read" page defines as a 30-day-vs-prior-30-day comparison — the same recent window the freshness story is trying to explain, not an outcome observed *after* a refresh. Was any part of this table re-checked against a later, independent period (did a page already showing "361+ days stale, refreshed" growth in one month keep growing in the next), or is the whole table a same-window association read as if it were forward-looking? And with a declining count of 1, the loudest cell falls below the paper's own stated minimum bucket size (n=50, or n=30 for cross-correlations, per the Methodology page) — should it be reported as a headline number at all under the paper's own rule, rather than just flagged as unstable in a caption?

**ML Appendix — "What Predicts Growth?" (p. 29).** A logistic regression reports 71% holdout accuracy on a sampled 61.8K-row active-content set across 57 brands, using an 80/20 split. The Methodology page names the split ratio but not whether it was random by row or grouped by brand.

My methodology question: this paper's own Finding #1 shows brands share consistent structural traits (word count, age, position) as a group. If the 80/20 split let rows from the same brand land in both train and holdout, part of that 71% could be the model recognizing a brand's house style rather than a signal that transfers to a brand it has never seen — the exact failure mode Section 2 below measures directly on my own model. Separately, the paper reports the 71% figure but not the growing/declining base rate of that 61.8K sample, so there's no way yet to tell how much of the 71% is above a naive majority-class guess.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 1 — receipts: spot-check the paper numbers I'm citing above, and the arithmetic behind the methodology question.

paper_totals = {
    "content_pieces": 341701,
    "brands": 57,
    "impressions": 469879632,
    "clicks": 1514819,
    "sessions": 1635404,
    "ai_sessions": 17344,
}
print("Verified totals I'm citing from the paper's Study Scope page:")
for k, v in paper_totals.items():
    print(f"  {k}: {v:,}")

growing_361plus, declining_361plus = 283, 1
print(f"\nFinding #4, 361+ day bucket: {growing_361plus}:{declining_361plus} "
      f"= {growing_361plus/declining_361plus:.0f}:1 growth-to-decline ratio")
print("Paper's own minimum bucket size (Methodology page): n=50 (n=30 for cross-correlations)")
print(f"Declining side of this bucket has n={declining_361plus} -> below both thresholds.")

print("\nML appendix growth model: 71% holdout accuracy, 80/20 split, grouping not stated,")
print("sample = 61.8K rows across 57 brands. Base rate of the growing/declining split: not reported.")

Verified totals I'm citing from the paper's Study Scope page:
  content_pieces: 341,701
  brands: 57
  impressions: 469,879,632
  clicks: 1,514,819
  sessions: 1,635,404
  ai_sessions: 17,344

Finding #4, 361+ day bucket: 283:1 = 283:1 growth-to-decline ratio
Paper's own minimum bucket size (Methodology page): n=50 (n=30 for cross-correlations)
Declining side of this bucket has n=1 -> below both thresholds.

ML appendix growth model: 71% holdout accuracy, 80/20 split, grouping not stated,
sample = 61.8K rows across 57 brands. Base rate of the growing/declining split: not reported.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model (`work/notebooks/w05_model.ipynb`) already used a client-grouped split — `GroupShuffleSplit` on `client_hash_id` — not a plain random split. Its own already-executed output (Section 2 of that notebook) shows: train clients 32, test clients 11, **client overlap between train and test: 0**, with `roc_auc` of 0.432 / 0.552 / 0.574 for the baseline rule / logistic regression / random forest respectively. So there's no dishonest "before" to recover from that notebook — it was built grouped from the start.

To make the before/after this assignment asks for concrete and something a reviewer can actually re-run, I rebuild the identical comparison — **naive random row split vs. client-grouped split**, same model family (random forest), same random seed — on the bundled local starter dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows, 32 clients) that ships in this repo. That notebook needs a live `HF_TOKEN` + Google Colab; this one runs fully offline, so every number below is something I (or a grader) can reproduce from a fresh clone with no credentials. Same client-grouping discipline, same leakage question, a dataset that's actually executable here.

- **BEFORE** — naive random row split (`train_test_split`, stratified only on the label). A client's pages can land in both train and test.
- **AFTER** — client-grouped holdout, the same logic `scripts/03_train_model.py` already uses: every client's rows land entirely in train or entirely in test.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 2 — load the starter dataset and rebuild the model feature vector
# (same feature/label definitions as scripts/01_prepare_features.py + scripts/ml_utils.py)

import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score

RANDOM_STATE = 42
REPO_URL = "https://github.com/krithi-ks/flyrankAI-ML.git"
REPO_DIR = Path("/content/flyrankAI-ML")  # Colab's cwd is /content, not the repo

# Try every path this notebook could plausibly be run from (Colab badge, local
# Jupyter opened at repo root, local Jupyter opened inside work/notebooks/).
candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    REPO_DIR / "data/raw/content_refresh_anonymized.csv",
]
RAW_PATH = next((p for p in candidate_paths if p.exists()), None)

if RAW_PATH is None:
    # Running in Colab (or any fresh environment): clone the repo so the bundled
    # starter CSV is available, then point at it there.
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    RAW_PATH = REPO_DIR / "data/raw/content_refresh_anonymized.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Could not find content_refresh_anonymized.csv anywhere I checked: {candidate_paths + [RAW_PATH]}"
    )

print(f"Loading starter dataset from: {RAW_PATH}")
raw = pd.read_csv(RAW_PATH)

NUMERIC_FILL_ZERO = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
for c in NUMERIC_FILL_ZERO:
    raw[c] = pd.to_numeric(raw[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

# Same population filter as the real pipeline: visible content, old enough to have a stable trend read.
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# The label trap: is_declining_label comes straight from trend_direction. Neither trend_direction
# nor trend_pct (which trend_direction is computed from) may appear in the features below.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
for c in MODEL_CATEGORICAL_FEATURES:
    df[c] = df[c].fillna("unknown").astype(str)

y = df["is_declining_label"]
numeric = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_enc = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES], prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([numeric.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)

print(f"Prepared rows: {len(df):,}  |  clients: {df['client_id'].nunique()}  |  overall base rate: {y.mean():.3f}")

Loading starter dataset from: /content/flyrankAI-ML/data/raw/content_refresh_anonymized.csv
Prepared rows: 30,000  |  clients: 32  |  overall base rate: 0.542


In [3]:
# Section 2 — the before/after comparison itself

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order][:k]
    return float(top.mean())

def fit_and_score(train_idx, test_idx, feature_frame=X):
    clf = RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
    )
    clf.fit(feature_frame.iloc[train_idx], y.iloc[train_idx])
    prob = clf.predict_proba(feature_frame.iloc[test_idx])[:, 1]
    pred = (prob >= 0.5).astype(int)
    metrics = {
        "roc_auc": roc_auc_score(y.iloc[test_idx], prob),
        "avg_precision": average_precision_score(y.iloc[test_idx], prob),
        "precision_at_50": precision_at_k(y.iloc[test_idx], prob, 50),
        "recall": recall_score(y.iloc[test_idx], pred),
        "f1": f1_score(y.iloc[test_idx], pred),
        "test_base_rate": y.iloc[test_idx].mean(),
    }
    return metrics, clf, prob

all_idx = np.arange(len(df))

# BEFORE: naive random row split — ignores which client a row belongs to.
train_before, test_before = train_test_split(
    all_idx, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
overlap_before = len(set(df.iloc[train_before]["client_id"]) & set(df.iloc[test_before]["client_id"]))
before_metrics, _, _ = fit_and_score(train_before, test_before)
before_metrics.update(client_overlap=overlap_before,
                       train_clients=df.iloc[train_before]["client_id"].nunique(),
                       test_clients=df.iloc[test_before]["client_id"].nunique())

# AFTER: client-grouped holdout — same logic as scripts/03_train_model.py's make_client_aware_split.
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled) * 0.2)))
test_clients_set = set(shuffled[:test_client_count])
test_mask = client_series.isin(test_clients_set).to_numpy()
train_after, test_after = all_idx[~test_mask], all_idx[test_mask]
overlap_after = len(set(df.iloc[train_after]["client_id"]) & set(df.iloc[test_after]["client_id"]))
after_metrics, rf_after, prob_after = fit_and_score(train_after, test_after)
after_metrics.update(client_overlap=overlap_after,
                      train_clients=df.iloc[train_after]["client_id"].nunique(),
                      test_clients=df.iloc[test_after]["client_id"].nunique())

assert overlap_after == 0, "Client leakage across the grouped split"

comparison = pd.DataFrame([
    {"split": "BEFORE - naive random row split", **before_metrics},
    {"split": "AFTER - client-grouped holdout", **after_metrics},
])
display(comparison.round(3))

,split,roc_auc,avg_precision,precision_at_50,recall,f1,test_base_rate,client_overlap,train_clients,test_clients
0,BEFORE - naive random row split,0.758,0.768,0.90,0.732,0.721,0.542,31,32,31
1,AFTER - client-grouped holdout,0.750,0.618,0.74,0.744,0.640,0.391,0,26,6


**Reading the before/after table:** the naive random split put 31 of the 32 clients in *both* train and test — the model could partly recognize a client it had already seen. That inflates `avg_precision` (0.618 → 0.768) and `precision_at_50` (0.740 → 0.900) the most, since those are exactly the ranking metrics a client-memorized model is best at gaming. `roc_auc` moves less (0.750 → 0.758) because it's a more global measure, which is itself a useful lesson — a healthy-looking AUC alone wouldn't have caught this; the client-overlap count and the ranking metrics did. The client-grouped split is the one I trust: it's the same discipline my Week-5 model already used, just demonstrated here with a visible before/after.

**Error examples — where the honest (client-grouped) model is actually wrong.** A metrics table hides what the mistakes look like, so here are real cases from the AFTER split above:

The two false negatives are low-impression, low-signal pages (1–3 impressions in 90 days) — the model has almost nothing to work with, so it defaults to a low decline-risk score even though they were in fact declining. That's a coverage gap, not a bad rule: a page this quiet doesn't give the model much to learn from.

The two false positives both belong to the *same* client (`client_f74efabef1`) — the model flagged both of that client's pages as high decline-risk (probability ≈ 0.74) even though neither actually declined. Since this client's pages sit in the held-out test set, the model has never seen this client's baseline behavior before, and it may be reading this client's normal position/freshness profile as risk signals learned from *other* clients. That's exactly the kind of client-specific pattern a client-grouped split is meant to surface — a naive random split would likely have hidden this by letting the model see this same client during training.

Feature importances on this same model: `days_with_impressions` and `log_impressions_90d` lead, followed by `avg_position` and `content_age_days` — visibility and position dominate, which matches the intuition behind both my baseline rule and the paper's own feature-importance appendix (position and impressions lead there too).

In [6]:
# Section 2 (continued) — real failure examples from the honest, client-grouped model

test_df = df.iloc[test_after].copy()
test_df["rf_prob"] = prob_after
test_df["rf_pred"] = (prob_after >= 0.5).astype(int)
test_df["actual"] = y.iloc[test_after].values

cols = ["content_id", "client_id", "impressions_90d", "avg_position",
        "days_since_last_update", "word_count", "rf_prob", "actual"]

false_negatives = test_df[(test_df["actual"] == 1) & (test_df["rf_pred"] == 0)] \
    .sort_values("rf_prob").head(2)
false_positives = test_df[(test_df["actual"] == 0) & (test_df["rf_pred"] == 1)] \
    .sort_values("rf_prob", ascending=False).head(2)

print("FALSE NEGATIVES (missed declines):")
display(false_negatives[cols])

print("\nFALSE POSITIVES (flagged, not actually declining):")
display(false_positives[cols])

importances = pd.Series(rf_after.feature_importances_, index=X.columns) \
    .sort_values(ascending=False).head(6)
print("\nTop 6 feature importances (client-grouped model):")
display(importances)

FALSE NEGATIVES (missed declines):


,content_id,client_id,impressions_90d,avg_position,days_since_last_update,word_count,rf_prob,actual
5770,content_28b4223f4e5f,client_98a3ab7c34,1,0.0,1,3109.0,0.079867,1
3879,content_34b14c00f80c,client_d4735e3a26,3,0.0,20,659.0,0.082196,1



FALSE POSITIVES (flagged, not actually declining):


,content_id,client_id,impressions_90d,avg_position,days_since_last_update,word_count,rf_prob,actual
23250,content_d2dffcc697a4,client_f74efabef1,5091,14.1,20,4496.0,0.737130,0
23559,content_00603b0349b4,client_f74efabef1,1076,25.6,20,2439.0,0.734944,0



Top 6 feature importances (client-grouped model):


,0
days_with_impressions,0.134951
log_impressions_90d,0.129377
avg_position,0.109203
content_age_days,0.092048
char_count,0.038676
age_tier_365+,0.036847


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist from the leakage skill against the feature set from Section 2 above (`MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES`, the same list `scripts/ml_utils.py` and my Week-5 notebook both use):

- **Label-derived features.** `is_declining_label` is built directly from `trend_direction` (`trend_direction == 'down'`), and `trend_direction` is itself computed from `trend_pct`. Neither field is in the feature matrix. The code cell below deliberately adds `trend_pct` back for one run — exactly the "train with vs. without the suspect" check the skill recommends — to prove the harness actually notices, then removes it again.
- **Decision-derived / ID fields.** `content_id` and `client_id` are pseudonyms used only to build the grouped split; neither appears as a model input.
- **Population selection.** The modeling frame keeps rows where `impressions_90d > 0` and `content_age_days >= 90`. Both conditions are measured at the same time as the features, not derived from whether the page later grew or declined, so this filter isn't smuggling in outcome-window information.
- **Base rate, printed next to its split.** Train-fold base rate and test-fold base rate for the client-grouped split are shown below — they don't match exactly, because they're built from different sets of clients. That's expected, and it's itself a small piece of evidence for why grouping by client (rather than assuming clients are interchangeable) is the right split.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 3 — deliberately add the suspect back in, watch the score jump, then remove it

X_leak = X.copy()
X_leak["trend_pct__SUSPECT"] = df["trend_pct"].values  # the field the label is built from

suspect_metrics, _, _ = fit_and_score(train_after, test_after, feature_frame=X_leak)

print(f"WITHOUT suspect (current pipeline, client-grouped split): ROC AUC = {after_metrics['roc_auc']:.3f}")
print(f"WITH suspect (trend_pct added back in):                   ROC AUC = {suspect_metrics['roc_auc']:.3f}")
print(f"Jump: {suspect_metrics['roc_auc'] - after_metrics['roc_auc']:+.3f}  "
      "-> the harness correctly flags this as leakage; trend_pct stays excluded.\n")

suspect_cols = ["trend_pct", "trend_direction", "is_declining_label", "content_id", "client_id"]
present = [c for c in suspect_cols if c in X.columns]
print("Suspect / ID columns present in the real feature matrix (should be empty):", present)

print(f"\nTrain-fold base rate (client-grouped): {y.iloc[train_after].mean():.3f}")
print(f"Test-fold base rate (client-grouped):  {y.iloc[test_after].mean():.3f}")

WITHOUT suspect (current pipeline, client-grouped split): ROC AUC = 0.750
WITH suspect (trend_pct added back in):                   ROC AUC = 1.000
Jump: +0.250  -> the harness correctly flags this as leakage; trend_pct stays excluded.

Suspect / ID columns present in the real feature matrix (should be empty): []

Train-fold base rate (client-grouped): 0.555
Test-fold base rate (client-grouped):  0.391


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest version (what a rushed status update might say):** "Our random forest model catches declining content with 90% precision in the top 50, so FlyRank editors can trust the top of the queue automatically." That number is real — it's the `precision_at_50` from the naive random split in Section 2 — but it only exists because 31 of 32 clients leaked across train and test. It's exactly the kind of score the leakage skill calls a confession, not an achievement, and I'm not keeping it.

**Rewritten in safe language:** in a client-grouped, held-out evaluation where no client appears in both train and test, the random forest model reached a measured `precision@50` of 0.740 against a held-out base rate of 0.391 — a directional improvement over the transparent baseline rule (`precision_at_50` of 0.240, per `outputs/model_report.md`). This is decision support for ordering which pages a human reviewer looks at first, not an automated action, and it should be re-checked against a new client cohort before it's trusted for anything beyond that.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4 — pull the exact numbers behind both versions of the claim, so the rewrite isn't hand-typed

bold_claim = (
    f"Our random forest model catches declining content with "
    f"{before_metrics['precision_at_50']*100:.0f}% precision in the top 50, "
    f"so FlyRank editors can trust the top of the queue automatically."
)
safe_claim = (
    f"In a client-grouped, held-out evaluation (client overlap = {after_metrics['client_overlap']}), "
    f"the random forest model reached a measured precision@50 of {after_metrics['precision_at_50']:.3f} "
    f"against a held-out base rate of {after_metrics['test_base_rate']:.3f} -- a directional improvement "
    f"over the transparent baseline rule (precision@50 of 0.240). Decision support for review "
    f"prioritization, not an automated action."
)

print("BOLD (naive-split, not trustworthy):\n ", bold_claim)
print("\nSAFE (client-grouped, what I'm actually claiming):\n ", safe_claim)

BOLD (naive-split, not trustworthy):
  Our random forest model catches declining content with 90% precision in the top 50, so FlyRank editors can trust the top of the queue automatically.

SAFE (client-grouped, what I'm actually claiming):
  In a client-grouped, held-out evaluation (client overlap = 0), the random forest model reached a measured precision@50 of 0.740 against a held-out base rate of 0.391 -- a directional improvement over the transparent baseline rule (precision@50 of 0.240). Decision support for review prioritization, not an automated action.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.